<a href="https://colab.research.google.com/github/BensonLing0925/Cuda_Kernel_Learning/blob/main/Day1_Kernel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!nvidia-smi

Fri Jun 19 05:14:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P0             26W /   70W |     139MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
pip install ninja

In [10]:
import torch
from torch.utils.cpp_extension import load_inline

# Check nvcc version to ensure it's available and compatible
!nvcc --version

cuda_src = r'''
__global__ void vadd(const float* a, const float* b, float* c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}
torch::Tensor vadd_cuda(torch::Tensor a, torch::Tensor b) {
    auto c = torch::empty_like(a);
    int n = a.numel(), threads = 256, blocks = (n + threads - 1) / threads;
    vadd<<<blocks, threads>>>(a.data_ptr<float>(), b.data_ptr<float>(), c.data_ptr<float>(), n);
    return c;
}
'''
mod = load_inline(name="vadd", cpp_sources="torch::Tensor vadd_cuda(torch::Tensor, torch::Tensor);",
                  cuda_sources=cuda_src,extra_cuda_cflags=["-arch=sm_75"],functions=["vadd_cuda"], verbose=True)

a = torch.randn(1_000_000, device="cuda"); b = torch.randn(1_000_000, device="cuda")
print("max error:", (mod.vadd_cuda(a, b) - (a + b)).abs().max().item())

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


KeyboardInterrupt: 

In [1]:
import torch
from torch.utils.cpp_extension import load_inline

# Check nvcc version to ensure it's available and compatible
!nvcc --version

cuda_src = r'''
__global__ void vadd(const float* a, const float* b, float* c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}
__global__ void matmul_naive(const float* A, const float* B, float* C, int M, int N, int K) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    // boundary checks
    if (row < M && col < N) {
      float sum = 0;
      for (int k = 0 ; k < K ; ++k) {
        sum += A[row*K+k]*B[k*N+col];
      }
      C[row*N+col]=sum;
    }
}
torch::Tensor vadd_cuda(torch::Tensor a, torch::Tensor b) {
    auto c = torch::empty_like(a);
    int n = a.numel(), threads = 256, blocks = (n + threads - 1) / threads;
    vadd<<<blocks, threads>>>(a.data_ptr<float>(), b.data_ptr<float>(), c.data_ptr<float>(), n);
    return c;
}
torch::Tensor matmul_naive_cuda(torch::Tensor a, torch::Tensor b) {
    int M = a.size(0), K = a.size(1), N = b.size(1);
    auto c = torch::empty({M, N}, a.options());
    dim3 threads(16, 16);
    dim3 blocks((N + 15) / 16, (M + 15) / 16);
    matmul_naive<<<blocks, threads>>>(a.data_ptr<float>(), b.data_ptr<float>(), c.data_ptr<float>(), M, N, K);
    return c;
}
'''
mod = load_inline(name="matmul_naive", cpp_sources="torch::Tensor matmul_naive_cuda(torch::Tensor a, torch::Tensor b);",
                  cuda_sources=cuda_src,extra_cuda_cflags=["-arch=sm_75"],functions=["matmul_naive_cuda"], verbose=True)

A = torch.randn(100, 100, device="cuda")
B = torch.randn(100, 100, device="cuda")
ref = A @ B
cuda_out = mod.matmul_naive_cuda(A, B)
ref = ref.to("cpu")
cuda_out = cuda_out.to("cpu")
print("max error:", (cuda_out - ref).abs().max().item())

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
max error: 1.52587890625e-05
